# Сегментация разливов на объединённом датасете




In [1]:
%pip install -q "ultralytics==8.4.188"

In [2]:
from datetime import datetime
from pathlib import Path
from zipfile import ZipFile

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Для обучения требуется GPU в среде Colab")
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")
drive_dir = Path("/content/drive/MyDrive/IndustrialVision")
dataset_name = "spill_mixed_yolo"
archive = Path("/content") / f"{dataset_name}.zip"
if not archive.is_file():
    archive = drive_dir / archive.name
if not archive.is_file():
    raise FileNotFoundError(f"Архив ожидается в /content или в {drive_dir}")

with ZipFile(archive) as zip_file:
    zip_file.extractall("/content")

data_yaml = Path("/content") / dataset_name / "data.yaml"
if not data_yaml.is_file():
    raise FileNotFoundError(f"Конфигурация датасета отсутствует: {data_yaml}")

run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output = drive_dir / "spill_runs" / dataset_name / run_id
output.mkdir(parents=True)
print("Результаты будут сохранены в", output)

GPU: Tesla T4
Mounted at /content/drive
Результаты будут сохранены в /content/drive/MyDrive/IndustrialVision/spill_runs/spill_mixed_yolo/20261122_124801_128300


In [3]:
import gc
import shutil

from ultralytics import YOLO

variants = ("n",)
train_options = {
    "data": str(data_yaml),
    "epochs": 60,
    "patience": 0,  # Отключаем раннюю остановку.
    "imgsz": 640,
    "batch": 8,
    "device": 0,
    "workers": 2,
    "seed": 42,
    "amp": True,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "cos_lr": True,
    "mosaic": 0.0,
    "scale": 0.2,
    "translate": 0.05,
    "hsv_s": 0.3,
    "hsv_v": 0.2,
    "project": str(output),
    "plots": True,
}

for variant in variants:
    name = f"yolo26{variant}-seg"
    model = YOLO(f"{name}.pt")
    metrics = model.train(name=name, **train_options)

    # Лучшие веса сохраняются под отдельным именем.
    best_file = output / f"spill_{name}.pt"
    shutil.copy2(model.trainer.best, best_file)
    print("Веса:", best_file)
    print(f"mAP50 масок: {metrics.seg.map50:.3f}")
    print(f"mAP50–95 масок: {metrics.seg.map:.3f}")

    del model, metrics
    gc.collect()
    torch.cuda.empty_cache()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
New https://pypi.org/project/ultralytics/8.4.216 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.188 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/spill_mixed_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, emb

In [4]:
from zipfile import ZIP_DEFLATED, ZipFile

from google.colab import files

download_zip = output / "spill_mixed_yolo26n_results.zip"
with ZipFile(download_zip, "w", compression=ZIP_DEFLATED) as bundle:
    for variant in variants:
        name = f"yolo26{variant}-seg"
        weights_name = f"spill_{name}.pt"
        bundle.write(output / weights_name, weights_name)
        last_weights = output / name / "weights" / "last.pt"
        bundle.write(last_weights, f"spill_{name}_last.pt")
        for filename in ("args.yaml", "results.csv"):
            report = Path(name) / filename
            bundle.write(output / report, report.as_posix())

print("Архив на Google Диске:", download_zip)
files.download(str(download_zip))

Архив на Google Диске: /content/drive/MyDrive/IndustrialVision/spill_runs/spill_mixed_yolo/20261122_124801_128300/spill_mixed_yolo26n_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>